In [ ]:
# ------------------------------------
# OLD CODE
# ------------------------------------
# import glob
# import numpy as np
# import librosa

# segment_dur = 2
# directories = ['guitar', 'non_guitar']
# def load_segments(directory):
#     segments = []
#     for path in glob.glob(f'{directory}/*.wav'):
#         signal, sr = librosa.load(path, sr=None, mono=True)
#         seg_len = sr * segment_dur
#         audio_segments = librosa.util.frame(signal, frame_length=seg_len, hop_length=seg_len, axis=0)
#         segments.append(audio_segments)
#     return np.concatenate(segments)

# guitar_segments = load_segments('guitar')
# non_guitar_segments = load_segments('non_guitar')
# def get_chords(path):

#     chords = []
#     with open(path) as f:
#         for line in f:      
#             start_s, end_s, chord = line.strip().split('\t')
#             chords.append(chord)
#     return chords


# guitar_chords = []
# for i in range(6):
#     guitar_chords += get_chords(f'guitar/guitar_annotation.lab')

# non_guitar_chords = []
# for i in range(10):
#     non_guitar_chords += get_chords(f'non_guitar/non_guitar_annotation.lab')

/Applications/anaconda3/envs/ai-py311/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, librosa, pandas as pd

sr = 16000
X_audio = np.load("data/processed/clips_16k.npy")
labels = pd.read_csv("data/processed/labels.csv")

chroma = librosa.feature.chroma_stft(y=X_audio, sr=sr, n_fft=8192, hop_length=4096)
X = np.concatenate([chroma.mean(axis=-1), chroma.std(axis=-1)], axis=1)
y = labels["original_label"].str.split("/").str[0].to_numpy()
# Labels: drop the inversion (e.g. 'C:maj/3' -> 'C:maj') -> 84 classes (12 roots x 7 qualities)
print(X.shape, y.shape, len(set(y)), 'classes')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

(7398, 24) (7398,) 84 classes


In [14]:
# sklearn: SVM (RBF kernel)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm = SVC(C=10).fit(X_train, y_train)
print('SVM test accuracy:', accuracy_score(y_test, svm.predict(X_test)))

SVM test accuracy: 0.8337837837837838


In [ ]:
# PyTorch: small MLP on the same features
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(y)
Xtr = torch.tensor(X_train, dtype=torch.float32)
Xte = torch.tensor(X_test, dtype=torch.float32)
ytr = torch.tensor(le.transform(y_train))
yte = torch.tensor(le.transform(y_test))

torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, len(le.classes_)),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        idx = perm[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr[idx]), ytr[idx])
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(Xte).argmax(1) == yte).float().mean().item()
        print(f'epoch {epoch + 1}: loss {loss.item():.3f}, test acc {acc:.3f}')

In [ ]:
# sklearn SVM on each feature set, with and without inversions
from sklearn.pipeline import make_pipeline

for name, F in feature_sets.items():
    for label_name, labels in [('no inv (84)', y_noinv), ('with inv (288)', y_full)]:
        clf = make_pipeline(StandardScaler(), SVC(C=10)).fit(F[train_idx], labels[train_idx])
        print(f'{name:15s} {label_name:15s} acc = {clf.score(F[test_idx], labels[test_idx]):.3f}')